# Create a Semantic Segmentation Table from Oxford-IIIT Pets

Ingest the [Oxford-IIIT Pets](https://www.robots.ox.ac.uk/~vgg/data/pets/) dataset into 3LC `Table`s for semantic segmentation, using the `semantic_segmentation` sample type. Each pet image ships with a trimap that we remap to three classes — `background`, `pet`, and `border` — and store compactly as run-length-encoded (RLE) per-class layers.

<!-- Tags: ["semantic-segmentation", "oxford-pets"] -->

Semantic segmentation assigns every pixel a class label, producing a dense `(H, W)` map that exhaustively partitions the image. The `semantic_segmentation` sample type stores this map in the instance-segmentation RLE wire format (one RLE per class present), so it renders in the 3LC Dashboard like any other segmentation while staying compact on disk — `background` (id 0) is implicit and recovered as the zero fill on read.

This is the *ingest* half of a pair: the companion notebook **`pytorch-oxford-pets-unet-training`** trains a small UNet on the tables created here and collects per-sample metrics back into 3LC.

## Project setup

In [ ]:
# Parameters
PROJECT_NAME = "3LC Tutorials - Oxford-IIIT Pets"
DATASET_NAME = "oxford-iiit-pets"
TABLE_NAME = "train"          # the train table; the val table is written as "val"
VAL_TABLE_NAME = "val"
DOWNLOAD_PATH = "../../transient_data"
N_TRAIN = 3000
N_VAL = 680
SEED = 42
INSTALL_DEPENDENCIES = True

## Install dependencies

In [ ]:
if INSTALL_DEPENDENCIES:
    %pip install -q 3lc torchvision matplotlib

## Imports

In [ ]:
from __future__ import annotations

import random
from pathlib import Path

import numpy as np
import tlc
from PIL import Image
from torchvision.datasets import OxfordIIITPet

from tlc.sample_types import SemanticSegmentation
from tlc.schemas import SemanticSegmentationRLESchema

## Download the dataset

`torchvision` handles the download. We only need it to fetch the files; afterwards we read the
images and trimaps straight from disk so the `Table` references the original image files by URL
(rather than re-encoding them).

In [ ]:
OxfordIIITPet(root=DOWNLOAD_PATH, split="trainval", target_types="segmentation", download=True)
data_root = Path(DOWNLOAD_PATH) / "oxford-iiit-pet"
print("Dataset at", data_root.resolve())

## Define the class universe

The trimap stores three pixel values — `1 = pet`, `2 = background`, `3 = border`. We remap them to a
0-based label map with `background = 0`. Two classes get special roles, declared on the schema so that
everything downstream reads the roles back rather than hard-coding ids:

- **background** (`id 0`) — the implicit fill. It is *not* stored on the wire; it is recovered as the
  zero region on read, which is what keeps storage compact.
- **border** (`id 2`) — tagged **void** (a "don't care" ring around each pet), so it can be excluded
  from the loss and from metrics like mean IoU.

In [ ]:
# Trimap pixel values: 1 = pet, 2 = background, 3 = border. Remap to 0-based with background = 0.
TRIMAP_REMAP = {2: 0, 1: 1, 3: 2}
SEGMENTATION_CLASSES = {0: "background", 1: "pet", 2: "border"}
BACKGROUND_ID = 0
VOID_ID = 2
SPECIES_CLASSES = ["cat", "dog"]

## Parse the annotations

`annotations/trainval.txt` has one line per image: `IMAGE CLASS-ID SPECIES BREED-ID`. We use it to
recover each image's breed (37 classes) and species (cat/dog), and to locate the paired trimap.

In [ ]:
def parse_annotations(root: Path) -> list[dict]:
    # Parse trainval.txt into per-sample dicts.
    samples = []
    for line in (root / "annotations" / "trainval.txt").read_text().splitlines():
        if line.startswith("#") or not line.strip():
            continue
        name, class_id, species, _breed_id = line.split()
        image_path = root / "images" / f"{name}.jpg"
        trimap_path = root / "annotations" / "trimaps" / f"{name}.png"
        if not image_path.exists() or not trimap_path.exists():
            continue
        samples.append(
            {
                "name": name,
                "image_path": image_path,
                "trimap_path": trimap_path,
                "breed": int(class_id) - 1,   # 0-based
                "species": int(species) - 1,  # 0: cat, 1: dog
            }
        )
    return samples


def breed_names(samples: list[dict]) -> list[str]:
    # Derive 0-indexed breed names from the image file names.
    names: dict[int, str] = {}
    for s in samples:
        names[s["breed"]] = s["name"].rsplit("_", 1)[0].replace("_", " ").lower()
    return [names[i] for i in range(max(names) + 1)]


samples = parse_annotations(data_root)
breeds = breed_names(samples)
print(f"Found {len(samples)} annotated samples across {len(breeds)} breeds")

## Convert a trimap to a `SemanticSegmentation`

`SemanticSegmentation` takes a dense integer mask plus the image dimensions and the background id. The
sample type takes care of encoding it as one RLE per present class.

In [ ]:
def load_segmentation(trimap_path: Path) -> SemanticSegmentation:
    trimap = np.asarray(Image.open(trimap_path))
    mask = np.zeros_like(trimap, dtype=np.int32)
    for src, dst in TRIMAP_REMAP.items():
        mask[trimap == src] = dst
    return SemanticSegmentation(
        image_width=trimap.shape[1],
        image_height=trimap.shape[0],
        mask=mask,
        background_id=BACKGROUND_ID,
    )

## Write the tables

The `segmentation` column uses `SemanticSegmentationRLESchema`, declaring the class names and the
`background` / `void` roles. We shuffle deterministically and write a `train` and a `val` table.

In [ ]:
def write_table(rows: list[dict], table_name: str) -> tlc.Table:
    writer = tlc.TableWriter(
        project_name=PROJECT_NAME,
        dataset_name=DATASET_NAME,
        table_name=table_name,
        schema={
            "image": tlc.schemas.ImageSchema(),
            "segmentation": SemanticSegmentationRLESchema(
                classes=SEGMENTATION_CLASSES, background=BACKGROUND_ID, void=VOID_ID
            ),
            "species": tlc.schemas.CategoricalLabelSchema(SPECIES_CLASSES),
            "breed": tlc.schemas.CategoricalLabelSchema(breeds),
        },
        if_exists="overwrite",
    )
    for s in rows:
        writer.add_row(
            {
                "image": tlc.Url(s["image_path"]).to_relative().to_str(),
                "segmentation": load_segmentation(s["trimap_path"]),
                "species": s["species"],
                "breed": s["breed"],
            }
        )
    table = writer.finalize()
    print(f"Wrote {table_name}: {len(table)} rows -> {table.url}")
    return table


random.Random(SEED).shuffle(samples)
train_rows = samples[:N_TRAIN]
val_rows = samples[N_TRAIN : N_TRAIN + N_VAL]

train_table = write_table(train_rows, TABLE_NAME)
val_table = write_table(val_rows, VAL_TABLE_NAME)

## Inspect a sample

Reading a row back hands you a `SemanticSegmentation` dataclass — confirming the round-trip through the
RLE wire format. We overlay the mask on the image to sanity-check the alignment.

In [ ]:
import matplotlib.pyplot as plt

sample = train_table[0]
seg = sample["segmentation"]
print(f"Round-trip type: {type(seg).__name__}, classes present: {seg.present_class_ids.tolist()}")

image = Image.open(tlc.Url(sample["image"]).to_absolute().to_str())
mask = seg.mask  # dense (H, W) label map

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
axes[0].imshow(image)
axes[0].set_title(f"{breeds[sample['breed']]} ({SPECIES_CLASSES[sample['species']]})")
axes[1].imshow(image)
axes[1].imshow(mask, alpha=0.5, cmap="viridis", interpolation="nearest")
axes[1].set_title("background / pet / border")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

## Next steps

The `train` and `val` tables are now in your 3LC project — open the project in the
[3LC Dashboard](https://dashboard.3lc.ai) to browse the images and their segmentations.

Continue with **`pytorch-oxford-pets-unet-training`** to train a UNet on these tables and collect
per-sample predictions, IoU, and embeddings back into a 3LC `Run`.